# 1. Pull MSA approval listings

Downloads each manufacturer's utility-approval listing and saves the raw response **untouched** to `data/raw/msa/YYYY-MM-DD/`. No parsing happens here, so if a parser breaks later the history is still intact.

| Source | Page | How it's fetched |
|---|---|---|
| ConnectDER | connectder.com/utility-approval-status | The site has a Vercel bot checkpoint, but its content comes from a public Prismic CMS API, which returns the same data as JSON |
| Enphase | enphase.com/.../iq-meter-collar/approvals | Plain HTML |
| Tesla | tesla.com/support/.../tesla-backup-switch | Plain HTML behind Akamai bot protection; needs a real (headed) Chromium window |

If one source fails, the others are still saved and the failure is logged. Each snapshot gets a `fetch_status.json` listing which sources succeeded.

**Parameter:** `catch_up=True` fetches only the sources not yet fetched successfully this calendar month (the scheduled run uses this so a missed or failed month is retried the next day).

In [ ]:
catch_up = False

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import requests

from common import MSA_SOURCES, RAW_MSA_DIR, get_logger, msa_sources_due

log = get_logger("pull_msa")

## Fetchers

In [ ]:
USER_AGENT = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/140.0 Safari/537.36")
ENPHASE_URL = "https://enphase.com/installers/storage/gen4/iq-meter-collar/approvals"
TESLA_URL = "https://www.tesla.com/support/energy/powerwall/learn/tesla-backup-switch"
CONNECTDER_API = "https://connectder.cdn.prismic.io/api/v2"
CONNECTDER_TYPES = ["utility_provider", "utility_install_type", "utility_approval_status"]


def fetch_connectder(out_dir):
    """All utility documents from ConnectDER's Prismic API, one JSON file per page."""
    session = requests.Session()
    ref = session.get(CONNECTDER_API, timeout=60).json()["refs"][0]["ref"]
    types = ",".join(f'"{t}"' for t in CONNECTDER_TYPES)
    paths, page, total_pages = [], 1, 1
    while page <= total_pages:
        resp = session.get(
            f"{CONNECTDER_API}/documents/search",
            params={"ref": ref, "q": f"[[any(document.type,[{types}])]]", "pageSize": 100, "page": page},
            timeout=60,
        )
        resp.raise_for_status()
        total_pages = resp.json()["total_pages"]
        path = out_dir / f"connectder_p{page}.json"
        path.write_bytes(resp.content)
        paths.append(path)
        page += 1
    return paths


def fetch_enphase(out_dir):
    resp = requests.get(ENPHASE_URL, headers={"User-Agent": USER_AGENT}, timeout=60)
    resp.raise_for_status()
    path = out_dir / "enphase.html"
    path.write_bytes(resp.content)
    return [path]


def load_tesla_page():
    """Akamai rejects requests, curl and headless Chromium. A headed Chromium
    with the automation flag disabled gets through; its window is parked
    off-screen. The first response is a JS challenge that reloads into the real
    page, so keep the last main-frame response and only accept it once it
    contains the listing."""
    from playwright.sync_api import sync_playwright

    documents = []
    with sync_playwright() as p:
        browser = p.chromium.launch(
            headless=False,
            args=["--disable-blink-features=AutomationControlled", "--window-position=-2400,-2400"],
        )
        try:
            page = browser.new_context(locale="en-US").new_page()
            page.on("response", lambda r: documents.append(r) if r.request.is_navigation_request()
                    and r.frame == page.main_frame else None)
            page.goto(TESLA_URL, wait_until="domcontentloaded", timeout=90000)
            try:
                page.wait_for_selector("text=Utilities That Have Approved Backup Switch",
                                       state="attached", timeout=60000)
            except Exception:
                statuses = [r.status for r in documents]
                raise RuntimeError(f"listing never loaded; document statuses {statuses} (likely Akamai bot block)")
            return documents[-1].body()
        finally:
            browser.close()


def fetch_tesla(out_dir):
    # Playwright's sync API can't run inside Jupyter's event loop, so the
    # browser runs in its own thread.
    with ThreadPoolExecutor(max_workers=1) as pool:
        body = pool.submit(load_tesla_page).result()
    if b"Backup Switch for Powerwall" not in body:
        raise RuntimeError("final response is missing the Powerwall listing")
    path = out_dir / "tesla.html"
    path.write_bytes(body)
    return [path]


FETCHERS = {"connectder": fetch_connectder, "enphase": fetch_enphase, "tesla": fetch_tesla}
assert set(FETCHERS) == set(MSA_SOURCES)

## Fetch

In [ ]:
snapshot_date = date.today().isoformat()
sources = msa_sources_due(snapshot_date) if catch_up else list(FETCHERS)
failed = []

if not sources:
    print(f"{snapshot_date}: all sources already fetched this month; nothing to do")
else:
    out_dir = RAW_MSA_DIR / snapshot_date
    out_dir.mkdir(parents=True, exist_ok=True)
    log.info("start  snapshot=%s  sources=%s", snapshot_date, ", ".join(sources))
    for name in sources:
        try:
            paths = FETCHERS[name](out_dir)
            log.info("OK      %-10s %s", name, ", ".join(f"{p.name} ({p.stat().st_size:,} B)" for p in paths))
        except Exception as e:
            failed.append(name)
            log.error("FAILED  %-10s %s: %s", name, type(e).__name__, e)

    # Merge with an earlier run from the same day so its successes are kept.
    status_file = out_dir / "fetch_status.json"
    ok = [n for n in sources if n not in failed]
    if status_file.exists():
        ok = sorted(set(ok) | (set(json.loads(status_file.read_text())["ok"]) - set(failed)))
    status_file.write_text(json.dumps({"snapshot_date": snapshot_date, "ok": ok, "failed": failed}, indent=1))
    log.info("done   %d/%d sources fetched%s", len(sources) - len(failed), len(sources),
             f"; FAILED: {', '.join(failed)}" if failed else "")

In [ ]:
# Fail the run (after saving what worked) so a scheduled run reports the error.
if failed:
    raise RuntimeError(f"failed to fetch: {', '.join(failed)}")